# Data Description
- Background: Unit 1.7 補充資料驗證：標線型人行道來源候選。提供機關：臺北市交通局交通管制工程處；官方來源：https://data.taipei/dataset/detail?id=d647b3af-4bbd-4e8f-b420-8f6c1bc1b597。原始來源對應與官方最新資源bytes完全相同為不同命題；除歷史界內包已逐byte雜湊對應外，未重新下載認證其他最新資源。
- date received: 學生於本單元提供；實際下載／收到日期未提供。首次本機檢視2026-11-26，不將今日當下載日。
- Path to data file: 官方來源及版本如上；精確本機容器如下，解壓至data/temp僅為衍生暫存。
  - `data/raw/(交工處)標線型人行道圖資_202610300914.zip`；4,487,594 bytes；SHA-256 `c17a853905ee242f88a9cbdefd4cf59130c08ab0eae40495365b6d36857a70a9`。
- Unit of observation: 混合標線面紀錄；未證人行道子集，不強造人行道特徵。
- Sample period: 2026-10-30快照；RDDATE僅現存紀錄歷史資格，不含拆除／遷移歷史；發布／更新：資源2026-10-30。ZIP時間、檔名時間、觀測／設置時間分開，不由檔名猜全年存量。
- Known issues: 實檔含多種LICODE並重現行穿線；缺確切人行道代碼篩選依據，不能全部43016面當人行道。缺失、重複、坐標及CRS由下方實讀輸出；沒有完整性證據的0僅為本檔未觀測。授權：政府資料開放授權條款第1版；依官方頁，來源與版本須保留。
- Definition for each variable: 下方逐欄型態、缺失、觀測域與定義表。已知官方欄意與本次診斷欄分開；未獲欄位碼表支持者明記未知，不依名稱猜設施單位。
- Data path: 上列每個獨立原始容器；實際讀取成員列於Summary Stats。
- Role: raw。
- Verification status: 以本Notebook乾淨kernel實際輸出與執行metadata為準；不認證已清理。
- SHA-256: 上列每檔固定綁定，載入前逐項重核。
- Next processing notebook: 本單元僅診斷，Chapter 2尚未建立；[驗證一](../code/taipei_supplementary_accident_boundary_validation.ipynb)、[驗證二](../code/taipei_transportation_infrastructure_validation.ipynb)。

Author: 詹明翰、謝子詮、曾翊涵；Created At / Last Modified At: 2026-11-26。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。共同作者與職務不代表已完成個別工作。本輪由Codex協助執行，成員驗收與審查尚待完成。


In [1]:
from pathlib import Path
import sys, hashlib, json, zipfile
import pandas as pd
from IPython.display import display, Markdown
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate/'AGENTS.md').is_file() and (candidate/'templates').is_dir():
        PROJECT_ROOT=candidate;break
else:raise FileNotFoundError('請從本研究工作包內執行')
def fingerprint(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_config.json')=='7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7'
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_support.py')=='1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
from unit17_validation_support import Context, show, safe_text
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',160)
ctx=Context(PROJECT_ROOT,'7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7')
print('Python:',sys.version,'Config:','7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7','Helper:','1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080')


Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)] Config: 7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7 Helper: 1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080


## Change for Your Own Data

使用已核對精確路徑及固定SHA-256；不選第一個glob或最新檔。CP950非法bytes只作可逆診斷；大檔分批讀取、OSM僅研究區2km bbox，保留全國母體數。

In [2]:
FAMILY='sidewalk'
INPUTS=ctx.family_items(FAMILY)
for item in INPUTS:ctx.file(item['path'])
frames=ctx.load_family(FAMILY)

## Summary

每個容器、實際成員與載入範圍分列；沒有將暫存解壓檔當原始下載資料。

In [3]:
show('固定輸入身分',pd.DataFrame([{k:v for k,v in r.items() if k not in ['members','encoding_checks']} for r in INPUTS]))
show('讀取範圍與CRS／編碼',pd.DataFrame(ctx.summaries[FAMILY]))

**固定輸入身分**

,family,path,bytes,sha256,status
0,sidewalk,data/raw/(交工處)標線型人行道圖資_202610300914.zip,4487594,c17a853905ee242f88a9cbdefd4cf59130c08ab0eae40495365b6d36857a70a9,present


**讀取範圍與CRS／編碼**

,path,member,total_features,loaded_rows,crs,types,bounds,invalid,null_geometry,empty
0,data/raw/(交工處)標線型人行道圖資_202610300914.zip,grapline_202610300914.shp,43016,43016,EPSG:3826,"{'Polygon': 42921, 'MultiPolygon': 95}","[296587.6529, 2762273.08262, 313676.8664, 2786850.235619]",0,0,0


### Sample

各實際讀取表前十列；全國事故為臺北篩選後第一當事人，不代表全國前十列。OSM為bbox空間子集。

In [4]:
for source,frame in frames.items():
    show(source,frame.drop(columns='geometry',errors='ignore').head(10),block=10)

**data/raw/(交工處)標線型人行道圖資_202610300914.zip::grapline_202610300914.shp**

,keyid,dxf_angle,rditcd,rdcode,licode,liarea,rddate,rdsncn,sirean,rdctnb,editby,rdlblg,rdlbwt,workc_keyi,streetview,streetvi_1,streetvi_2,streetvi_3,streetvi_4,workc_in_u,workc_in_t,edittime,zlocation,parefe,walkerroad,proj_name,proj_no,rpfrom,rparea,rplocation,rpdate,walkerro_1,walkerstat,roadid,_source_row,_source_path,_source_member
fid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,8580,0.0,0.0,None,08,0.0,2013-05-16,0101,01,NaN,董映琳,0.0,0.0,0.0,1.352802e+07,2.890309e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-05,01,FIX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,1,data/raw/(交工處)標線型人行道圖資_202610300914.zip,grapline_202610300914.shp
1,9913,0.0,0.0,None,A8,0.0,2008-01-10,0101,01,NaN,董映琳,0.0,0.0,0.0,1.352881e+07,2.889571e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-07,01,FIX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,2,data/raw/(交工處)標線型人行道圖資_202610300914.zip,grapline_202610300914.shp
2,21505,0.0,0.0,None,A8,0.0,2022-03-16,NaN,01,NaN,董映琳,10.0,40.0,18069.0,0.000000e+00,0.000000e+00,0.000000,0.000000,0.00,299.0,2022-04-19,2023-01-07,01,FIX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,3,data/raw/(交工處)標線型人行道圖資_202610300914.zip,grapline_202610300914.shp
3,11265,0.0,0.0,None,A8,0.0,1999-06-24,NaN,02,NaN,董映琳,0.0,0.0,0.0,1.352949e+07,2.890053e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-07,01,FIX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,4,data/raw/(交工處)標線型人行道圖資_202610300914.zip,grapline_202610300914.shp
4,38288,0.0,NaN,None,08,0.0,2023-01-07,NaN,NaN,NaN,董映琳,0.0,0.0,NaN,1.352965e+07,2.890278e+06,-31.042314,-0.055751,2.98,NaN,NaT,2023-01-07,01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,5,data/raw/(交工處)標線型人行道圖資_202610300914.zip,grapline_202610300914.shp
5,11276,0.0,0.0,None,08,0.0,2008-01-10,0101,01,NaN,董映琳,0.0,0.0,0.0,1.352944e+07,2.890252e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-07,01,FIX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,6,data/raw/(交工處)標線型人行道圖資_202610300914.zip,grapline_202610300914.shp
6,38289,0.0,NaN,None,08,0.0,2023-01-07,NaN,NaN,NaN,董映琳,0.0,0.0,NaN,1.352930e+07,2.890274e+06,-31.042314,-0.055751,2.98,NaN,NaT,2023-01-07,01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,7,data/raw/(交工處)標線型人行道圖資_202610300914.zip,grapline_202610300914.shp
7,38292,0.0,NaN,None,A8,0.0,2023-01-07,NaN,NaN,NaN,董映琳,0.0,0.0,NaN,1.352955e+07,2.889712e+06,-31.042314,-0.055751,2.98,NaN,NaT,2023-01-07,01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,8,data/raw/(交工處)標線型人行道圖資_202610300914.zip,grapline_202610300914.shp
8,27098,0.0,0.0,None,07,0.0,2008-01-10,NaN,NaN,NaN,董映琳,0.0,0.0,0.0,1.352917e+07,2.889391e+06,-31.042314,-0.055751,2.98,0.0,NaT,2023-01-07,01,FIX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,9,data/raw/(交工處)標線型人行道圖資_202610300914.zip,grapline_202610300914.shp


### Summary Stats

ZIP逐成員CRC、配套、原始WKT、全部欄型態／缺失／定義、重複與數值／類別／時間診斷。保留原值及bytes。

In [5]:
_ = ctx.raw_report(FAMILY,crc=True)

,source_metadata_and_limits
title,標線型人行道來源候選
provider,臺北市交通局交通管制工程處
url,https://data.taipei/dataset/detail?id=d647b3af-4bbd-4e8f-b420-8f6c1bc1b597
publication,資源2026-10-30
period,2026-10-30快照；RDDATE僅現存紀錄歷史資格，不含拆除／遷移歷史
limitation,實檔含多種LICODE並重現行穿線；缺確切人行道代碼篩選依據，不能全部43016面當人行道
license,政府資料開放授權條款第1版；依官方頁，來源與版本須保留


,family,path,bytes,sha256,status
0,sidewalk,data/raw/(交工處)標線型人行道圖資_202610300914.zip,4487594,c17a853905ee242f88a9cbdefd4cf59130c08ab0eae40495365b6d36857a70a9,present


**ZIP原始成員清單；非解壓暫存資料來源**

,member,bytes,compressed,crc32,zip_datetime
0,grapline_202610300914.cpg,5,7,0e813c50,"[2026, 10, 30, 9, 14, 48]"
1,grapline_202610300914.dbf,54588426,1601110,3004b4ac,"[2026, 10, 30, 9, 14, 52]"
2,grapline_202610300914.prj,399,256,90c78104,"[2026, 10, 30, 9, 14, 48]"
3,grapline_202610300914.shp,6034424,2761677,3dd6195b,"[2026, 10, 30, 9, 14, 52]"
4,grapline_202610300914.shx,344228,123892,e751d06f,"[2026, 10, 30, 9, 14, 52]"


ZIP 全成員CRC通過: data/raw/(交工處)標線型人行道圖資_202610300914.zip


**實際讀取範圍／編碼／CRS／全國或地區計數**

... (13 outputs left out)

原始屬性完整列重複（不含診斷追溯欄） 0


**類別前12項：licode**

,licode,N,ratio_of_loaded_rows
0,07,10868,0.252650
1,A8,8482,0.197182
2,08,7405,0.172145
3,21,4513,0.104914
4,10,2227,0.051771
5,18,2012,0.046773
6,B8,1831,0.042566
7,A0,907,0.021085
8,12,818,0.019016
9,16,490,0.011391


,field,missing_or_unparseable,pre1950_placeholder,eligible_survivors_2024,eligible_survivors_2025,post2025
0,rddate,5299,84,37618,37626,7


本文件只確認資料身分及可讀性；語義／日期不足者限制用途，不強行納入正式Dataset B3。人類驗收仍待完成。

In [6]:
for item in INPUTS:ctx.file(item['path'])
print('原始容器hash重核通過；正式分析資料輸出0。')

原始容器hash重核通過；正式分析資料輸出0。
